# 03_backtest_strategies.ipynb - Backtest de Estrategias Generadas

**Objetivo**: Cargar estrategias generadas en `02_strategy_generator.ipynb` y ejecutar backtest completo en el universo admitido.

Flujo:
1. 01_eda.ipynb → EDA
2. batch_calibrate.py → Csl + BP calibrados
3. 02_strategy_generator.ipynb → Genera estrategias → `data/strategies_generated.json`
4. **03_backtest_strategies.ipynb ← ESTE: Backteste estrategias**
5. 04_robustez.ipynb → Tests robustez
6. 05_validacion.ipynb → Validación final

## 1. Imports

In [ ]:
import sys
import json
import pandas as pd
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

from config.settings import DATA, FEATURES, RISK, BACKTEST
from src.data import load_ohlc_from_yfinance
from src.features import add_all_features
from src.risk import run_backtest_loop, calculate_performance_metrics

# Placeholder: Aquí importaremos las funciones del generador cuando lo implementes
# from src.strategy_builder import build_signal_from_strategy

print("Imports listos. Esperando strategies_generated.json...")

## 2. Cargar Estrategias Generadas

In [ ]:
strategies_path = REPO_ROOT / "data" / "strategies_generated.json"

try:
    with open(strategies_path, 'r') as f:
        strategies = json.load(f)
    print(f"Cargadas {len(strategies)} estrategias generadas.")
except FileNotFoundError:
    print("⚠️ Archiro strategies_generated.json no existe aún.")
    print("   Runs 02_strategy_generator.ipynb primero.")
    strategies = []

## 3. Backtest por Estrategia y Activo

In [ ]:
if not strategies:
    print("Sin estrategias para backtestear.")
else:
    all_trades = []
    all_metrics = []
    
    for strat in strategies:
        strat_name = strat.get('name', 'unnamed')
        print(f"\n=== Estrategia: {strat_name} ===")
        
        # Cargar universo admitido
        admitted_df = pd.read_csv(REPO_ROOT / "data" / "universe_admitted.csv")
        
        for _, row in admitted_df.iterrows():
            ticker = row['ticker']
            csl = row['csl']
            
            try:
                df = load_ohlc_from_yfinance(ticker, period=DATA.period, interval=DATA.interval)
                df = add_all_features(df, atr_period=FEATURES.atr_period, sma_period=FEATURES.sma_period, roc_period=FEATURES.roc_period)
                
                # TODO: IMPORTAR TU FUNCIÓN DE GENERACIÓN DE SEÑALES
                # signal = build_signal_from_strategy(df, strat)
                
                # Placeholder:…] placeholder
                signal = pd.Series(False, index=df.index)  # ¡NO EJECUTOR! Solo estructura.
                
                # if signal.sum() > 0:
                #     trades = run_backtest_loop(df, signal, csl=csl, ...)
                #     metrics = calculate_performance_metrics(trades, ...)
                #     [...] asigchar a all_trades/all_metrics
                
                print(f"  {ticker}: ☕ Pending signal implementation")
                
            except Exception as e:
                print(f"  {ticker}: Error {e}")
    
    # Guardar resultados (placeholder)
    out_dir = REPO_ROOT / "data"
    # if all_trades:
    #     pd.concat(all_trades, ignore_index=True).to_csv(out_dir / "trades_backtest.csv", index=False)
    #     pd.DataFrame(all_metrics).to_csv(out_dir / "metrics_backtest.csv", index=False)

## 4. Próximos pasos

1. Implementar `02_strategy_generator.ipynb` para generar `strategies_generated.json`
2. Crear `src/strategy_builder.py` con lógica para convertir estrategia → señal booleana
3. Completar este notebook con la llamada a esa función
4. Ejecutar y ver resultados